# Tutorial: Arabic Sentence Chunking

This notebook demonstrates how to chunk Arabic text into sentences. We will cover two approaches:
1. **General Approach**: Using `nltk` for standard Arabic text.
2. **Specific Approach**: Using Regular Expressions to handle Quranic waqf (stop) marks as sentence boundaries.

Audience:
- Developers working with Arabic NLP or Quranic text analysis.

Prerequisites:
- Basic Python knowledge.
- `nltk` and `re` libraries.

Learning goals:
- Understand how to tokenize Arabic sentences using standard libraries.
- Learn how to handle specialized text like the Quran using regex markers.

## Setup

We'll start by importing the necessary libraries. If you don't have `nltk` installed, you can install it via `pip install nltk`.

In [1]:
import re
import nltk

# Download necessary tokenizer data
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

print("Setup complete.")

Setup complete.


## Fetching Text from API

Instead of hardcoding the text, we will fetch it from the **Al-Quran Cloud API**. We will use Ayat al-Kursi (2:255), which corresponds to ayah index 262 in the global count.

In [2]:
import requests

# Main version (Uthmani)
# Surah 2, Ayah 255 (Ayat al-Kursi)
ayah_url = "https://api.alquran.cloud/v1/ayah/2:255/quran-uthmani"
response = requests.get(ayah_url)
ayah_data = response.json()

# Fetch simple-clean version for normalization/search
simple_clean_url = "https://api.alquran.cloud/v1/ayah/2:255/quran-simple-clean"
simple_clean_response = requests.get(simple_clean_url)
simple_clean_text = simple_clean_response.json()["data"]["text"]

if response.status_code == 200:
    ayah_info = ayah_data["data"]
    text = ayah_info["text"]
    edition_info = ayah_info["edition"]
    hizb_quarter = ayah_info["hizbQuarter"]
    print(f"Fetched Ayah {ayah_info['number']} ({edition_info['identifier']})")
    print(f"Text Length: {len(text)}")
else:
    print(f"Failed to fetch ayah. Status code: {response.status_code}")

Fetched Ayah 262 (quran-uthmani)
Text Length: 427


## Fetching Context for LLM

To provide context for an LLM, we fetch the entire text of the corresponding `hizbQuarter`.

In [3]:
hizb_url = f"https://api.alquran.cloud/v1/hizbQuarter/{hizb_quarter}/quran-uthmani"
hizb_response = requests.get(hizb_url)
hizb_data = hizb_response.json()

if hizb_response.status_code == 200:
    # Combine all ayahs in the hizbQuarter for context
    context_text = " ".join([a["text"] for a in hizb_data["data"]["ayahs"]])
    print(f"Fetched Context (Hizb Quarter {hizb_quarter}) - Total Length: {len(context_text)} characters")
else:
    context_text = ""
    print("Failed to fetch context.")

Fetched Context (Hizb Quarter 17) - Total Length: 3208 characters


## Method 1: Using NLTK (General Approach)

`nltk.sent_tokenize` is great for standard prose but might struggle with Quranic text if common punctuation (like periods) is missing.

In [4]:
nltk_sentences = nltk.sent_tokenize(text)

print(f"NLTK found {len(nltk_sentences)} sentence(s):")
for i, sent in enumerate(nltk_sentences, 1):
    print(f"{i}. {sent.strip()}\n")

NLTK found 1 sentence(s):
1. ٱللَّهُ لَآ إِلَٰهَ إِلَّا هُوَ ٱلْحَىُّ ٱلْقَيُّومُ ۚ لَا تَأْخُذُهُۥ سِنَةٌۭ وَلَا نَوْمٌۭ ۚ لَّهُۥ مَا فِى ٱلسَّمَٰوَٰتِ وَمَا فِى ٱلْأَرْضِ ۗ مَن ذَا ٱلَّذِى يَشْفَعُ عِندَهُۥٓ إِلَّا بِإِذْنِهِۦ ۚ يَعْلَمُ مَا بَيْنَ أَيْدِيهِمْ وَمَا خَلْفَهُمْ ۖ وَلَا يُحِيطُونَ بِشَىْءٍۢ مِّنْ عِلْمِهِۦٓ إِلَّا بِمَا شَآءَ ۚ وَسِعَ كُرْسِيُّهُ ٱلسَّمَٰوَٰتِ وَٱلْأَرْضَ ۖ وَلَا يَـُٔودُهُۥ حِفْظُهُمَا ۚ وَهُوَ ٱلْعَلِىُّ ٱلْعَظِيمُ



## Method 2: Regex for Quranic Waqf Marks (Specific Approach)

In the Quran, waqf marks like ۚ (Jim), ۗ (Qalat), and ۖ (Salat) are the actual indicators for pauses or sentence endings. We can use regex to split the text based on these characters.

In [5]:
# Define the waqf marks as split patterns
# \u06d6 = ۖ (Salat)
# \u06d7 = ۗ (Qalat)
# \u06da = ۚ (Jim)

waqf_pattern = r"([ۚۗۖ])"

# Split while keeping the delimiter
parts = re.split(waqf_pattern, text)

# Reconstruct sentences by joining the mark back to the preceding text
regex_sentences = []
for i in range(0, len(parts)-1, 2):
    regex_sentences.append(parts[i] + parts[i+1])

# Add the last part if it exists
if len(parts) % 2 != 0 and parts[-1].strip():
    regex_sentences.append(parts[-1])

print(f"Regex found {len(regex_sentences)} sentence(s) based on waqf marks:")
for i, sent in enumerate(regex_sentences, 1):
    print(f"{i}. {sent.strip()}\n")

Regex found 9 sentence(s) based on waqf marks:
1. ٱللَّهُ لَآ إِلَٰهَ إِلَّا هُوَ ٱلْحَىُّ ٱلْقَيُّومُ ۚ

2. لَا تَأْخُذُهُۥ سِنَةٌۭ وَلَا نَوْمٌۭ ۚ

3. لَّهُۥ مَا فِى ٱلسَّمَٰوَٰتِ وَمَا فِى ٱلْأَرْضِ ۗ

4. مَن ذَا ٱلَّذِى يَشْفَعُ عِندَهُۥٓ إِلَّا بِإِذْنِهِۦ ۚ

5. يَعْلَمُ مَا بَيْنَ أَيْدِيهِمْ وَمَا خَلْفَهُمْ ۖ

6. وَلَا يُحِيطُونَ بِشَىْءٍۢ مِّنْ عِلْمِهِۦٓ إِلَّا بِمَا شَآءَ ۚ

7. وَسِعَ كُرْسِيُّهُ ٱلسَّمَٰوَٰتِ وَٱلْأَرْضَ ۖ

8. وَلَا يَـُٔودُهُۥ حِفْظُهُمَا ۚ

9. وَهُوَ ٱلْعَلِىُّ ٱلْعَظِيمُ



## Preparing Final Records (DB & LLM Ready)

We now combine the chunked sentences with metadata and context into a final JSON structure.

In [6]:
final_records = []

for i, sentence in enumerate(regex_sentences, 1):
    record = {
        "id": f"quran_2_255_sent_{i}",
        "text": sentence.strip(),
        "source_type": "quran",
        "metadata": {
            "ayah_number": ayah_info["number"],
            "surah": ayah_info["surah"],
            "edition": edition_info,
            "version_type": "quran-uthmani",
            "simple_clean_text": simple_clean_text,
            "source_url": ayah_url,
            "other_editions": [] 
        },
        "llm_context": context_text,
        "chunk_index": i
    }
    final_records.append(record)

print(f"Prepared {len(final_records)} records for the 'sentence' table (Source: Quran).")

Prepared 9 records for the 'sentence' table (Source: Quran).


## Inserting Data into SurrealDB

Finally, we insert the prepared records into the `ayah_segment` table in SurrealDB.

In [7]:
from surrealdb import Surreal

def insert_to_surreal():
    # Connection details
    db_url = "ws://surrealdb:8000/rpc"
    
    with Surreal(db_url) as db:
        db.signin({"user": "root", "pass": "root"})
        db.use("main", "main")
        
        for record in final_records:
            content = {
                "text": record["text"],
                "source_type": record["source_type"],
                "simple_clean_text": record["metadata"]["simple_clean_text"],
                "llm_context": record["llm_context"],
                "chunk_index": record["chunk_index"],
                "ayah_number": record["metadata"]["ayah_number"],
                "surah_number": record["metadata"]["surah"]["number"],
                "surah_name": record["metadata"]["surah"]["name"],
                "version_type": record["metadata"]["version_type"],
                "edition_identifier": record["metadata"]["edition"]["identifier"],
                "source_url": record["metadata"]["source_url"],
                "other_editions": record["metadata"]["other_editions"]
            }
            
            record_id = f"sentence:{record['id']}"
            db.create(record_id, content)
            print(f"Inserted: {record_id}")

# Run the insertion
insert_to_surreal()
print("All records inserted into 'sentence' table successfully!")

Inserted: sentence:quran_2_255_sent_1
Inserted: sentence:quran_2_255_sent_2
Inserted: sentence:quran_2_255_sent_3
Inserted: sentence:quran_2_255_sent_4
Inserted: sentence:quran_2_255_sent_5
Inserted: sentence:quran_2_255_sent_6
Inserted: sentence:quran_2_255_sent_7
Inserted: sentence:quran_2_255_sent_8
Inserted: sentence:quran_2_255_sent_9
All records inserted into 'sentence' table successfully!


## Conclusion

As seen above, `nltk` might treat the entire verse as a single sentence if it doesn't recognize the waqf marks. The **Regex approach** is much more effective for religious or specialized Arabic texts.